# MX conv on the NPE emulator — two-layer example

Standalone demo of `npe_mx_emu`.  The library emulates the hardware pipe
(TRU -> BCU -> PCU); calibration comes from **observers** collecting
block-amax statistics over data samples:

- `MXObserver(mode="mx")` anchors the smallest non-zero block exponent at 0
  (`calibrate_offset = mantissa_offset - floor(log2(a_min))`; block exponent
  extraction is `floor(log2)` -- Q1, HW-confirmed -- so the block-max mantissa
  fills `[64, 127]` and no block's exponent is ever negative).
- `MXObserver(mode="input")` uses the integer-sensor range-fit rule
  (`calibrate = mantissa_offset - ceil(log2(a_max))`, zero shared exponent).
- bias is put in accumulator units `2**(x_cal + w_cal)` by `build_c_struct`.
- raw int output has scale `2**-(2 * mantissa_offset)` (the HW default),
  exposed as `layer.output_scale_log2`.


In [1]:
import os
import sys
# make npe_mx_emu importable whether jupyter runs from the repo root or from EL_nb/
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import torch
import torch.nn as nn

from npe_mx_emu import LayerMXQParams, CalibratedMXConv2d, MXObserver


## Reference float network and input

Same dimensions as the original notebook experiment (1->2->2 channels, 32x32, K=3).


In [2]:
torch.manual_seed(1)

B, C_in, C_mid, C_out, H, W = 1, 1, 2, 2, 32, 32
K, S, P = 3, 1, 0

torch_conv1 = nn.Conv2d(C_in, C_mid, K, stride=S, padding=P)
torch_conv2 = nn.Conv2d(C_mid, C_out, K, stride=S, padding=P)

x = torch.rand(B, C_in, H, W) - 0.5

ref_out = torch_conv2(torch_conv1(x))


## Layer 1 — the integer input layer

Collect statistics from samples (here a single sample; loop over a
calibration set in real use).  The input layer uses `mode="input"` and
`is_input_layer=True` (zero shared exponent, raw int storage).

Activations are observed per W-block; weights are observed per **flattened kernel** (`per_row_flatten=True`), mirroring the HW data-consumption order (algo table Q9).

**Q3 rounding rule (HW-confirmed):** `x * 2^cal -> round` happens ONLY here,
at the first (integer) layer.  Intermediate layers skip any pre-round -- the
PCU already drains integers and its `shift_right` (`e_min_layer` sum)
realigns the scale between layers, so the MX quantizer's own per-block
truncate is the only integer-ization downstream.  The BCU shifters being
unsigned / left-only is exactly why operands must sit at non-negative
exponents in the first place.


In [3]:
obs_x1 = MXObserver(mode="input"); obs_x1.update(x)
obs_w1 = MXObserver(mode="mx");    obs_w1.update(torch_conv1.weight, per_row_flatten=True)  # weights are blocked per flattened kernel (Q9)

x_cal_1 = obs_x1.calibrate_offset()   # range-fit: 2**7 for [-0.5, 0.5) data
w_cal_1 = obs_w1.calibrate_offset()   # exponent-anchored: >= 2**7, no negative exps
print(f"layer1: x_cal={x_cal_1}, w_cal={w_cal_1}  "
      f"(w1 e_min={obs_w1.e_min_layer()}, w1 max_block_exp={obs_w1.max_block_exponent(w_cal_1)})")

qp1 = LayerMXQParams(x_cal=x_cal_1, w_cal=w_cal_1, is_input_layer=True)
layer1 = CalibratedMXConv2d.from_torch_conv(torch_conv1, qp1)
layer1.calibrate()

out1_raw = layer1(x)                     # raw integer accumulator output
out1_f = out1_raw / 2.0 ** layer1.output_scale_log2

ref1 = torch_conv1(x)
err1 = (out1_f - ref1).abs()
print(f"layer1 output scale: 2**-{layer1.output_scale_log2}")
print(f"layer1 max abs err: {err1.max().item():.6f}")
print("out1_f[0,:,0,0]:", out1_f[0, :, 0, 0])
print("ref1  [0,:,0,0]:", ref1[0, :, 0, 0])
print("out1_f[0,:,-1,-1]:", out1_f[0, :, -1, -1])
print("ref1  [0,:,-1,-1]:", ref1[0, :, -1, -1])



layer1: x_cal=7, w_cal=8  (w1 e_min=-2, w1 max_block_exp=1)
layer1 output scale: 2**-12
layer1 max abs err: 0.014729
out1_f[0,:,0,0]: tensor([0.4299, 0.1382])
ref1  [0,:,0,0]: tensor([0.4292, 0.1356], grad_fn=<SelectBackward0>)
out1_f[0,:,-1,-1]: tensor([0.3928, 0.1521])
ref1  [0,:,-1,-1]: tensor([0.3968, 0.1530], grad_fn=<SelectBackward0>)


In [4]:
# Q9 self-check: the observer's blocking must equal the quantizer's blocking
qp_bs = qp1.block_size
import math as _m
flat = torch_conv1.weight.reshape(torch_conv1.weight.shape[0], -1)
exp_obs_blocks = flat.shape[0] * _m.ceil(flat.shape[1] / qp_bs)
q_blocks = layer1.mx_conv.mx_weight.data.shape[0] * layer1.mx_conv.mx_weight.data.shape[1]
print(f"weight {tuple(torch_conv1.weight.shape)} -> flat per OC {flat.shape[1]} elems")
print(f"observer-derived blocks (per flattened kernel): {exp_obs_blocks}")
print(f"quantizer blocks (mx_weight):                    {q_blocks}")
assert exp_obs_blocks == q_blocks
print("Q9 ok: block partitions match")



weight (2, 1, 3, 3) -> flat per OC 9 elems
observer-derived blocks (per flattened kernel): 2
quantizer blocks (mx_weight):                    2
Q9 ok: block partitions match


## Pipeline walk-through: one output element, stage by stage

Trace a single output value `(oc=0, oh=0, ow=0)` of layer 1 through the
NPE pipe, with real numbers from the run above.


### Stage 0 — quantize the input

The input quantizer stores raw ints with a zero shared exponent
(`input_mx_quantizer`); `x_cal` came from `MXObserver(mode="input")`.


In [5]:
from npe_mx_emu import input_mx_quantizer, align_mx_to_elements, tru_a, tru_w, build_c_struct

x_int = (x * 2.0 ** qp1.x_cal).round()
mx_in = input_mx_quantizer(x_int, block_size=qp1.block_size, calibrate_offset=qp1.x_cal)
print(f"x_cal = {qp1.x_cal}  ->  x in [-0.5, 0.5) stored as ints in [-64, 64)")
print("x_int block 0      :", x_int[0, 0, 0, :8].int(), "...")
print("mx.data block 0    :", mx_in.data[0, 0, 0, 0, :8].int(), "...")
print("mx.shared_exp blk 0:", mx_in.shared_exp[0, 0, 0, 0].item(), "  (zero: integer input layer)")

mx_w = layer1.mx_conv.mx_weight
print(f"w_cal = {qp1.w_cal}  (anchored so no block exponent is negative)")
print("w exps per OC      :", mx_w.shared_exp.tolist())


x_cal = 7  ->  x in [-0.5, 0.5) stored as ints in [-64, 64)
x_int block 0      : tensor([ 39,  20, -41,  42,  39,  57, -36, -11], dtype=torch.int32) ...
mx.data block 0    : tensor([ 39,  20, -41,  42,  39,  57, -36, -11], dtype=torch.int32) ...
mx.shared_exp blk 0: 0.0   (zero: integer input layer)
w_cal = 8  (anchored so no block exponent is negative)
w exps per OC      : [[1], [0]]


### Stage 1 — TRU: reorder the activation line and the kernel


In [6]:
xd, xe = align_mx_to_elements(mx_in)
a_data, a_exp = tru_a(xd, xe, line=0, S_h=1, S_w=1, K_w=3, K_h=3, out_width=30, batch=0)
w_data, w_exp = tru_w(mx_w)
print("act patch (ow=0)   :", a_data[0].tolist())
print("act exp  (ow=0)   :", a_exp[0].tolist())
print("wgt flat (oc=0)   :", w_data[0].tolist())
print("wgt exp  (oc=0)   :", w_exp[0].tolist())


act patch (ow=0)   : [39, 20, -41, 21, -48, 13, 64, -40, 34]
act exp  (ow=0)   : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
wgt flat (oc=0)   : [21, -18, -8, 20, -40, 25, -8, 21, 5]
wgt exp  (oc=0)   : [1, 1, 1, 1, 1, 1, 1, 1, 1]


### Stage 2 — BCU: per-cycle multiply / shift / accumulate

Each cycle: `shifted = floor(mant_a * mant_w * 2**(exp_a + exp_w))`,
accumulated in order into `p_sum`.


In [7]:
from npe_mx_emu import bcu


pm = a_data[0] * w_data[0]
ce = a_exp[0] + w_exp[0]
sh = (pm * torch.exp2(ce.float())).floor()
ps = torch.zeros(())
for l in range(9):
    ps += sh[l]
    print(f"cycle {l}: mant={pm[l].item():>7.0f}  exp_sum={ce[l].item():>3.0f}  "
          f"shifted={sh[l].item():>10.0f}  acc={ps.item():>11.0f}")

p_sum = bcu(a_data, a_exp, w_data[0], w_exp[0])
print("bcu p_sum[0]         :", p_sum[0].item())


cycle 0: mant=    819  exp_sum=  1  shifted=      1638  acc=       1638
cycle 1: mant=   -360  exp_sum=  1  shifted=      -720  acc=        918
cycle 2: mant=    328  exp_sum=  1  shifted=       656  acc=       1574
cycle 3: mant=    420  exp_sum=  1  shifted=       840  acc=       2414
cycle 4: mant=   1920  exp_sum=  1  shifted=      3840  acc=       6254
cycle 5: mant=    325  exp_sum=  1  shifted=       650  acc=       6904
cycle 6: mant=   -512  exp_sum=  1  shifted=     -1024  acc=       5880
cycle 7: mant=   -840  exp_sum=  1  shifted=     -1680  acc=       4200
cycle 8: mant=    170  exp_sum=  1  shifted=       340  acc=       4540
bcu p_sum[0]         : 4540.0


### Stage 3 — PCU: bias (accumulator units) -> e_min shift -> round -> clamp

The PCU adds the bias (pre-scaled to accumulator units `2**(x_cal+w_cal)`,
= `2**b_cal`), shifts by the *sum* of the two operands' `e_min_layer`
(+ static shift = 0), rounds, and clamps to the int activation range.


In [8]:
cs = layer1.mx_conv.c_struct_arr[0]
e_min = layer1.mx_conv.mx_weight.e_min_layer + mx_in.e_min_layer
pla = p_sum[0] * cs.scale + cs.bias
shifted = (pla * 2.0 ** (cs.shift + e_min)).floor().round()
out00 = torch.clamp(shifted, min=cs.min_val, max=cs.max_val)
print(f"bias (acc units, 2**{qp1.b_cal})   : {cs.bias.item():.0f}")
print(f"e_min_layer sum (w + x)         : {int(e_min)}")
print(f"p_sum + bias                    : {pla.item():.0f}")
print(f"after e_min shift + round       : {out00.item():.0f}")
print(f"layer raw out[0,0,0,0]          : {out1_raw[0,0,0,0].item():.0f}  (match)")
print(f"float = raw / 2**{qp1.output_scale_log2}        : {out00.item() / 2.0 ** qp1.output_scale_log2:.6f}")
print(f"torch ref  conv1(x)[0,0,0,0]    : {ref1[0,0,0,0].item():.6f}")


bias (acc units, 2**15)   : 9548
e_min_layer sum (w + x)         : -3
p_sum + bias                    : 14088
after e_min shift + round       : 1761
layer raw out[0,0,0,0]          : 1761  (match)
float = raw / 2**12        : 0.429932
torch ref  conv1(x)[0,0,0,0]    : 0.429243


## Layer 2 — intermediate MX layer

Observe the float layer-1 output (in real use, feed the calibration set
through layer 1 and `update()` per sample).

As in layer 1: activations per W-block, weights per flattened kernel (`per_row_flatten=True`).


In [9]:
obs_x2 = MXObserver(mode="mx"); obs_x2.update(out1_f)
obs_w2 = MXObserver(mode="mx"); obs_w2.update(torch_conv2.weight, per_row_flatten=True)  # weights are blocked per flattened kernel (Q9)

x_cal_2 = obs_x2.calibrate_offset()
w_cal_2 = obs_w2.calibrate_offset()
print(f"layer2: x_cal={x_cal_2}, w_cal={w_cal_2}  "
      f"(x2 max_block_exp={obs_x2.max_block_exponent(x_cal_2)}, "
      f"w2 max_block_exp={obs_w2.max_block_exponent(w_cal_2)})")

qp2 = LayerMXQParams(x_cal=x_cal_2, w_cal=w_cal_2)
layer2 = CalibratedMXConv2d.from_torch_conv(torch_conv2, qp2)
layer2.calibrate()

out2_raw = layer2(out1_f)                # input is a plain float tensor, as with nn.Conv2d
out2_f = out2_raw / 2.0 ** layer2.output_scale_log2

err2 = (out2_f - ref_out).abs()
print(f"layer2 max abs err: {err2.max().item():.6f}")
print("out2_f  [0,:,-1,-1]:", out2_f[0, :, -1, -1])
print("ref_out [0,:,-1,-1]:", ref_out[0, :, -1, -1])



layer2: x_cal=8, w_cal=8  (x2 max_block_exp=2, w2 max_block_exp=0)
layer2 max abs err: 0.013109
out2_f  [0,:,-1,-1]: tensor([-0.0850, -0.0833])
ref_out [0,:,-1,-1]: tensor([-0.0877, -0.0847], grad_fn=<SelectBackward0>)


## Observer over multiple samples (EMA option)

The observer accumulates across `update()` calls.  `reduction="ema"`
smooths batch-to-batch variation but may break the no-negative-exponent
guarantee on unseen data; `minmax` (default) is the safe choice.


In [10]:
obs = MXObserver(mode="mx", reduction="ema", ema_momentum=0.5)
for i in range(4):
    obs.update(torch.rand(1, 1, 8, 8) - 0.5)
print(f"ema over {obs.n_samples} samples: cal={obs.calibrate_offset()}, "
      f"e_min={obs.e_min_layer()}, max_block_exp={obs.max_block_exponent(obs.calibrate_offset())}")


ema over 4 samples: cal=7, e_min=-1, max_block_exp=0


## Format comparison: e4bm8 vs e6bm8 vs e8bm8

Format names: `e{#exponent_bits}b{#mantissa_bits}`. All current configs
have an 8-bit mantissa, so `mantissa_offset` and the default output scale
(`2**12`) are identical; what changes is the exponent window (15 / 63 /
255). The observer raising "exceeds the window" is the signal to pick a
wider-exponent format.


In [11]:
for mt in ("mx-e4bm8", "mx-e6bm8", "mx-e8bm8"):
    ox, ow = MXObserver(mode="mx", mx_type=mt), MXObserver(mode="mx", mx_type=mt)
    ox.update(x); ow.update(torch_conv1.weight, per_row_flatten=True)
    q = LayerMXQParams(x_cal=ox.calibrate_offset(), w_cal=ow.calibrate_offset(),
                       mx_type=mt, is_input_layer=True)
    l = CalibratedMXConv2d.from_torch_conv(torch_conv1, q)
    l.calibrate()
    o = l(x) / 2.0 ** l.output_scale_log2
    print(f"{mt}: x_cal={q.x_cal:>2}, w_cal={q.w_cal:>2}, "
          f"max_block_exp={ow.max_block_exponent(q.w_cal):>2}, "
          f"max abs err={(o - ref1).abs().max().item():.6f}")


mx-e4bm8: x_cal= 7, w_cal= 8, max_block_exp= 1, max abs err=0.014729
mx-e6bm8: x_cal= 7, w_cal= 8, max_block_exp= 1, max abs err=0.014729
mx-e8bm8: x_cal= 7, w_cal= 8, max_block_exp= 1, max abs err=0.014729
